In [1]:
!pip install pymupdf reportlab pillow

In [2]:
import os
import re
import json
import math
from dataclasses import dataclass, asdict, field
from pathlib import Path
from typing import List, Dict, Tuple

import fitz  # PyMuPDF
from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch, cm
from reportlab.lib.enums import TA_LEFT, TA_CENTER
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    Image,
    PageBreak,
    Table,
    TableStyle,
    KeepTogether
)
from reportlab.lib.utils import ImageReader


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
THERMAL_PDF = "/content/drive/MyDrive/Dataset/Company/Thermal Images.pdf"
INSPECTION_PDF = "/content/drive/MyDrive/Dataset/Company/Sample Report.pdf"

OUTPUT_DIR = "ddr_output"
THERMAL_IMG_DIR = "thermal_page_images"
INSPECTION_IMG_DIR = "inspection_page_images"

OUTPUT_JSON = "structured_ddr.json"
OUTPUT_PDF = "generated_ddr.pdf"

EXPORT_ALL_PAGE_IMAGES = True
MAX_IMAGE_WIDTH = 5.8 * inch
MAX_IMAGE_HEIGHT = 3.8 * inch

In [5]:
@dataclass
class ImageRef:
    source_file: str
    page_number: int
    image_path: str
    caption: str = ""
    area_guess: str = ""
    image_kind: str = "generic"   # thermal | inspection | generic


@dataclass
class ThermalObservation:
    page_number: int
    thermal_image_name: str = "Not Available"
    date: str = "Not Available"
    hotspot_c: str = "Not Available"
    coldspot_c: str = "Not Available"
    raw_text: str = ""
    page_image_path: str = ""
    area_guess: str = ""


@dataclass
class AreaObservation:
    impacted_area_no: str
    area_name: str
    negative_observation: str = "Not Available"
    positive_observation: str = "Not Available"
    probable_root_cause: List[str] = field(default_factory=list)
    severity_level: str = "Not Available"
    severity_reasoning: str = "Not Available"
    recommended_actions: List[str] = field(default_factory=list)
    additional_notes: List[str] = field(default_factory=list)
    missing_or_unclear_information: List[str] = field(default_factory=list)
    supporting_checklist: List[str] = field(default_factory=list)
    related_summary_points: List[str] = field(default_factory=list)
    related_thermal_pages: List[ThermalObservation] = field(default_factory=list)
    related_images: List[ImageRef] = field(default_factory=list)


@dataclass
class PropertyContext:
    customer_name: str = "Not Available"
    mobile: str = "Not Available"
    email: str = "Not Available"
    address: str = "Not Available"
    property_age_years: str = "Not Available"
    floors: str = "Not Available"
    inspection_date: str = "Not Available"
    inspected_by: str = "Not Available"
    property_type: str = "Not Available"
    impacted_rooms: List[str] = field(default_factory=list)


@dataclass
class DDRResult:
    property_context: PropertyContext
    property_issue_summary: List[str]
    area_wise_observations: List[AreaObservation]
    probable_root_cause_summary: List[str]
    additional_notes: List[str]
    missing_or_unclear_information: List[str]
    conflicts: List[str]


# ============================================================
# UTILITIES
# ============================================================
def ensure_dir(path: str) -> None:
    Path(path).mkdir(parents=True, exist_ok=True)


def clean_text(text: str) -> str:
    if not text:
        return ""
    text = text.replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)
    return text.strip()


def normalize_for_match(text: str) -> str:
    text = (text or "").lower().strip()
    text = re.sub(r"[^a-z0-9 ]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text


def safe_get(pattern: str, text: str, flags: int = re.IGNORECASE) -> str:
    if not text:
        return "Not Available"
    m = re.search(pattern, text, flags)
    if not m:
        return "Not Available"
    return clean_text(m.group(1))


def unique_preserve_order(items: List[str]) -> List[str]:
    seen = set()
    out = []
    for item in items:
        item = clean_text(item)
        if not item:
            continue
        key = normalize_for_match(item)
        if key not in seen:
            seen.add(key)
            out.append(item)
    return out


def dedupe_images(images: List[ImageRef]) -> List[ImageRef]:
    seen = set()
    out = []
    for img in images:
        key = (img.source_file, img.page_number, img.image_path)
        if key not in seen:
            seen.add(key)
            out.append(img)
    return out


def html_escape(text: str) -> str:
    return (
        (text or "")
        .replace("&", "&amp;")
        .replace("<", "&lt;")
        .replace(">", "&gt;")
    )

In [6]:
def read_pdf_pages(pdf_path: str) -> List[Dict]:
    doc = fitz.open(pdf_path)
    pages = []
    for i, page in enumerate(doc):
        pages.append({
            "page_number": i + 1,
            "text": clean_text(page.get_text("text")),
            "page": page,
        })
    return pages


def export_page_renderings(pdf_path: str, output_dir: str, prefix: str) -> Dict[int, str]:
    ensure_dir(output_dir)
    doc = fitz.open(pdf_path)
    saved = {}
    for i, page in enumerate(doc):
        pix = page.get_pixmap(matrix=fitz.Matrix(2, 2), alpha=False)
        out_path = os.path.join(output_dir, f"{prefix}_page_{i+1:03d}.png")
        pix.save(out_path)
        saved[i + 1] = out_path
    return saved


In [7]:
def extract_property_context(inspection_pages: List[Dict]) -> PropertyContext:
    page1 = inspection_pages[0]["text"] if len(inspection_pages) >= 1 else ""
    page3 = inspection_pages[2]["text"] if len(inspection_pages) >= 3 else ""

    ctx = PropertyContext(
        customer_name=safe_get(r"Customer Name\s*(.*?)\s*Mobile:", page1, re.IGNORECASE | re.DOTALL),
        mobile=safe_get(r"Mobile:\s*(.*?)\s*Email:", page1, re.IGNORECASE | re.DOTALL),
        email=safe_get(r"Email:\s*(.*?)\s*Address:", page1, re.IGNORECASE | re.DOTALL),
        address=safe_get(r"Address:\s*(.*?)\s*Property Age", page1, re.IGNORECASE | re.DOTALL),
        property_age_years=safe_get(r"Property Age \(In years\):\s*(.*?)\s*Floors:", page1, re.IGNORECASE | re.DOTALL),
        floors=safe_get(r"Floors:\s*(.*?)\s*Previous Structural audit done", page1, re.IGNORECASE | re.DOTALL),
        inspection_date=safe_get(r"Inspection Date and Time:\s*(.*?)\s*Inspected By:", page1, re.IGNORECASE | re.DOTALL),
        inspected_by=safe_get(r"Inspected By:\s*(.*?)\s*Property Type:", page1, re.IGNORECASE | re.DOTALL),
        property_type=safe_get(r"Property Type:\s*(.*)", page1),
    )

    impacted_rooms_raw = safe_get(
        r"Impacted Areas/Rooms\s*(.*?)\s*Impacted Area",
        page3,
        re.IGNORECASE | re.DOTALL
    )
    if impacted_rooms_raw != "Not Available":
        impacted_rooms_raw = impacted_rooms_raw.replace("\n", " ")
        rooms = [x.strip(" ,") for x in impacted_rooms_raw.split(",") if x.strip()]
        ctx.impacted_rooms = rooms

    return ctx


In [8]:
AREA_BLOCK_PATTERN = re.compile(
    r"Impacted Area\s*(\d+)\s*"
    r"Negative side Description\s*(.*?)\s*"
    r"Negative side photographs\s*(.*?)\s*"
    r"Positive side Description\s*(.*?)\s*"
    r"Positive side photographs",
    re.IGNORECASE | re.DOTALL,
)


def infer_area_name_from_negative_obs(text: str) -> str:
    t = normalize_for_match(text)
    mapping = [
        ("master bedroom", "Master Bedroom"),
        ("common bedroom", "Bedroom"),
        ("bedroom", "Bedroom"),
        ("hall", "Hall"),
        ("kitchen", "Kitchen"),
        ("parking", "Parking Area"),
        ("common bathroom", "Common Bathroom"),
        ("bathroom", "Bathroom"),
        ("passage", "Passage Area"),
        ("external wall", "External Wall"),
    ]
    for key, value in mapping:
        if key in t:
            return value
    return "Not Available"


def extract_impacted_area_blocks(
    inspection_pages: List[Dict],
    inspection_img_map: Dict[int, str]
) -> List[AreaObservation]:
    combined = "\n".join(p["text"] for p in inspection_pages[:8])

    matches = list(AREA_BLOCK_PATTERN.finditer(combined))
    areas: List[AreaObservation] = []

    for m in matches:
        area_no = clean_text(m.group(1))
        negative_desc = clean_text(m.group(2))
        positive_desc = clean_text(m.group(4))
        area_name = infer_area_name_from_negative_obs(negative_desc)

        area = AreaObservation(
            impacted_area_no=area_no,
            area_name=area_name,
            negative_observation=negative_desc if negative_desc else "Not Available",
            positive_observation=positive_desc if positive_desc else "Not Available",
        )
        areas.append(area)

    if not areas:
        segments = re.split(r"Impacted Area\s*(\d+)", combined, flags=re.IGNORECASE)
        for i in range(1, len(segments), 2):
            area_no = clean_text(segments[i])
            body = segments[i + 1]

            neg = safe_get(
                r"Negative side Description\s*(.*?)\s*Negative side photographs",
                body,
                re.IGNORECASE | re.DOTALL
            )
            pos = safe_get(
                r"Positive side Description\s*(.*?)\s*Positive side photographs",
                body,
                re.IGNORECASE | re.DOTALL
            )

            areas.append(
                AreaObservation(
                    impacted_area_no=area_no,
                    area_name=infer_area_name_from_negative_obs(neg),
                    negative_observation=neg,
                    positive_observation=pos,
                )
            )

    for idx, area in enumerate(areas, start=1):
        likely_page = 3 + math.floor((idx - 1) / 2)
        if likely_page in inspection_img_map:
            area.related_images.append(
                ImageRef(
                    source_file=INSPECTION_PDF,
                    page_number=likely_page,
                    image_path=inspection_img_map[likely_page],
                    caption=f"Inspection evidence page {likely_page}",
                    area_guess=area.area_name,
                    image_kind="inspection",
                )
            )

    return areas


In [9]:
SUMMARY_ROW_PATTERN = re.compile(
    r"(\d+)\s+(Observed .*?)\s+(\d+\.\d+)\s+(Observed .*?)(?=(?:\n\d+\s+Observed)|$)",
    re.IGNORECASE | re.DOTALL,
)


def extract_summary_pairs(inspection_pages: List[Dict]) -> Dict[str, Dict[str, str]]:
    summary_page_text = ""
    for p in inspection_pages:
        if "SUMMARY TABLE" in p["text"].upper():
            summary_page_text = p["text"]
            break

    summary_map: Dict[str, Dict[str, str]] = {}
    if not summary_page_text:
        return summary_map

    rows = SUMMARY_ROW_PATTERN.findall(summary_page_text)
    for point_no, neg, pos_no, pos in rows:
        summary_map[clean_text(point_no)] = {
            "negative_summary": clean_text(neg),
            "positive_summary": clean_text(pos),
            "positive_point_no": clean_text(pos_no),
        }
    return summary_map

In [10]:
def extract_checklist_evidence(inspection_pages: List[Dict]) -> List[str]:
    full_text = "\n".join(p["text"] for p in inspection_pages[:12])
    evidence = []

    phrases = [
        "Leakage during:",
        "Leakage due to concealed plumbing",
        "Leakage due to damage in Nahani trap/Brickbat coba under",
        "Gaps/Blackish dirt Observed in tile joints",
        "Gaps around Nahani Trap Joints",
        "Loose Plumbing joints/rust around joints and edges",
        "Condition of cracks observed on RCC Column and Beam",
        "Are there any major or minor cracks observed over external",
        "Algae fungus and Moss observed on external wall?",
    ]

    for phrase in phrases:
        m = re.search(re.escape(phrase), full_text, re.IGNORECASE)
        if m:
            start = max(0, m.start() - 40)
            end = min(len(full_text), m.end() + 160)
            evidence.append(clean_text(full_text[start:end]))

    return unique_preserve_order(evidence)



In [11]:
def extract_thermal_observations(
    thermal_pages: List[Dict],
    thermal_img_map: Dict[int, str]
) -> List[ThermalObservation]:
    observations: List[ThermalObservation] = []

    for p in thermal_pages:
        text = p["text"]
        if "Thermal image" not in text and "Hotspot" not in text:
            continue

        obs = ThermalObservation(
            page_number=p["page_number"],
            thermal_image_name=safe_get(r"Thermal image\s*:\s*(.*?)\s*Device", text, re.IGNORECASE | re.DOTALL),
            date=safe_get(r"(\d{2}/\d{2}/\d{2,4})", text),
            hotspot_c=safe_get(r"Hotspot\s*:\s*([0-9.]+\s*°?C)", text),
            coldspot_c=safe_get(r"Coldspot\s*:\s*([0-9.]+\s*°?C)", text),
            raw_text=text,
            page_image_path=thermal_img_map.get(p["page_number"], ""),
        )
        observations.append(obs)

    return observations

In [12]:
AREA_KEYWORDS = {
    "Hall": ["hall"],
    "Bedroom": ["bedroom", "common bedroom"],
    "Master Bedroom": ["master bedroom", "mb"],
    "Kitchen": ["kitchen"],
    "Parking Area": ["parking"],
    "Common Bathroom": ["common bathroom"],
    "Bathroom": ["bathroom", "wc"],
    "Passage Area": ["passage"],
    "External Wall": ["external wall"],
}


def score_area_match(area: AreaObservation, thermal_obs: ThermalObservation) -> int:
    score = 0

    area_text = normalize_for_match(
        f"{area.area_name} {area.negative_observation} {area.positive_observation}"
    )
    thermal_text = normalize_for_match(thermal_obs.raw_text)

    for key in AREA_KEYWORDS.get(area.area_name, []):
        if key in thermal_text:
            score += 5
        if key in area_text:
            score += 1

    if "skirting" in area_text:
        score += 2
    if "damp" in area_text or "damness" in area_text or "seepage" in area_text:
        score += 2

    try:
        area_no = int(area.impacted_area_no)
        if abs(thermal_obs.page_number - area_no) <= 2:
            score += 2
    except Exception:
        pass

    if thermal_obs.coldspot_c != "Not Available":
        score += 1

    return score



In [13]:
def derive_root_causes(area: AreaObservation) -> List[str]:
    text = normalize_for_match(
        f"{area.negative_observation} {area.positive_observation} "
        f"{' '.join(area.supporting_checklist)} {' '.join(area.related_summary_points)}"
    )

    causes = []

    if any(k in text for k in ["tile joint", "gap", "nahani", "hollowness"]):
        causes.append("Likely water ingress through failed or open tile joints in the adjoining wet area")

    if any(k in text for k in ["plumbing", "outlet leakage", "concealed plumbing", "rust around joints"]):
        causes.append("Possible concealed or local plumbing leakage contributing to moisture spread")

    if any(k in text for k in ["external wall crack", "external wall", "crack", "moss", "algae"]):
        causes.append("Likely water ingress through cracks or weather-exposed external wall surfaces")

    if any(k in text for k in ["parking", "ceiling", "below flat", "below wc"]):
        causes.append("Leakage may be traveling downward from the wet area or floor above")

    if any(k in text for k in ["skirting", "efflorescence", "wall damness", "dampness"]):
        causes.append("Moisture movement through masonry or plaster is causing visible dampness at the lower wall level")

    if not causes:
        causes.append("Probable root cause could not be confirmed from the available inspection evidence")

    return unique_preserve_order(causes)


def assess_severity(area: AreaObservation) -> Tuple[str, str]:
    text = normalize_for_match(
        f"{area.negative_observation} {area.positive_observation} "
        f"{' '.join(area.supporting_checklist)} {' '.join(area.related_summary_points)}"
    )

    high_triggers = [
        "seepage", "parking", "ceiling", "outlet leakage",
        "plumbing issue", "external wall crack", "efflorescence"
    ]
    medium_triggers = [
        "dampness", "hollowness", "gap", "tile joint", "wall damness"
    ]

    high_score = sum(1 for x in high_triggers if x in text)
    med_score = sum(1 for x in medium_triggers if x in text)

    if high_score >= 2:
        return (
            "High",
            "Severity is marked High because the evidence suggests active or recurring moisture movement, multiple supporting defect indicators, or leakage affecting adjacent or below areas."
        )
    if med_score >= 1 or high_score == 1:
        return (
            "Medium",
            "Severity is marked Medium because visible dampness or related source-side defects are present, but the documents do not clearly confirm major structural damage or uncontrolled live leakage at this exact point."
        )
    return (
        "Low",
        "Severity is marked Low because the available evidence shows limited or mild visible impact and does not clearly indicate active severe leakage."
    )


def derive_recommended_actions(area: AreaObservation) -> List[str]:
    text = normalize_for_match(
        f"{area.negative_observation} {area.positive_observation} "
        f"{' '.join(area.supporting_checklist)} {' '.join(area.related_summary_points)}"
    )

    actions = []

    if any(k in text for k in ["tile joint", "gap", "nahani", "hollowness"]):
        actions.append("Repair and re-grout the affected tile joints after proper cleaning and preparation")

    if any(k in text for k in ["plumbing", "outlet leakage", "concealed plumbing"]):
        actions.append("Check the nearby plumbing line, outlet, and joints for hidden leakage and repair defective points")

    if any(k in text for k in ["external wall", "crack", "moss", "algae"]):
        actions.append("Seal external wall cracks and apply a suitable exterior waterproofing treatment")

    if any(k in text for k in ["efflorescence", "dampness", "skirting"]):
        actions.append("Remove damaged plaster or paint only after the source of moisture is controlled")

    if "parking" in text or "ceiling" in text:
        actions.append("Inspect the corresponding wet area above and verify whether moisture is migrating downward")

    if not actions:
        actions.append("Carry out a focused site reinspection because the available evidence is not sufficient for a precise action plan")

    return unique_preserve_order(actions)


In [14]:
def attach_summary_to_areas(areas: List[AreaObservation], summary_map: Dict[str, Dict[str, str]]) -> None:
    for area in areas:
        item = summary_map.get(area.impacted_area_no)
        if item:
            if item.get("negative_summary") and item["negative_summary"] != "Not Available":
                area.related_summary_points.append(item["negative_summary"])
            if item.get("positive_summary") and item["positive_summary"] != "Not Available":
                area.related_summary_points.append(item["positive_summary"])


def attach_checklists_to_areas(areas: List[AreaObservation], checklist_evidence: List[str]) -> None:
    for area in areas:
        area_text = normalize_for_match(
            f"{area.area_name} {area.negative_observation} {area.positive_observation}"
        )

        for ev in checklist_evidence:
            nev = normalize_for_match(ev)

            if (
                ("tile joint" in nev and any(x in area_text for x in ["bathroom", "kitchen", "bedroom", "hall"]))
                or ("concealed plumbing" in nev and any(x in area_text for x in ["bathroom", "parking", "ceiling"]))
                or ("external wall" in nev and "external" in area_text)
                or ("cracks" in nev and ("external" in area_text or "wall" in area_text))
                or ("moss" in nev and "external" in area_text)
            ):
                area.supporting_checklist.append(ev)

        area.supporting_checklist = unique_preserve_order(area.supporting_checklist)


def attach_thermal_to_areas(areas: List[AreaObservation], thermal_obs_list: List[ThermalObservation]) -> None:
    for area in areas:
        scored = sorted(
            [(score_area_match(area, t), t) for t in thermal_obs_list],
            key=lambda x: x[0],
            reverse=True,
        )

        selected = [t for s, t in scored[:2] if s >= 3]
        area.related_thermal_pages.extend(selected)

        for t in selected:
            if t.page_image_path:
                area.related_images.append(
                    ImageRef(
                        source_file=THERMAL_PDF,
                        page_number=t.page_number,
                        image_path=t.page_image_path,
                        caption=f"Thermal evidence page {t.page_number}: hotspot {t.hotspot_c}, coldspot {t.coldspot_c}",
                        area_guess=area.area_name,
                        image_kind="thermal",
                    )
                )

        area.related_images = dedupe_images(area.related_images)


def fill_reasoning(areas: List[AreaObservation]) -> None:
    for area in areas:
        area.probable_root_cause = derive_root_causes(area)
        level, reason = assess_severity(area)
        area.severity_level = level
        area.severity_reasoning = reason
        area.recommended_actions = derive_recommended_actions(area)

        if not area.related_images:
            area.missing_or_unclear_information.append("Image Not Available")

        if not area.related_thermal_pages:
            area.missing_or_unclear_information.append("Thermal mapping to this area is Not Available")

        if area.area_name == "Not Available":
            area.missing_or_unclear_information.append("Area name could not be confidently identified from the report")

        area.missing_or_unclear_information = unique_preserve_order(area.missing_or_unclear_information)


def build_property_summary(areas: List[AreaObservation]) -> List[str]:
    summaries = []

    damp_areas = [a.area_name for a in areas if "damp" in normalize_for_match(a.negative_observation)]
    damp_areas = unique_preserve_order(damp_areas)
    if damp_areas:
        summaries.append(f"Visible dampness is reported in multiple areas, including {', '.join(damp_areas)}.")

    if any("parking" in normalize_for_match(a.area_name) or "parking" in normalize_for_match(a.negative_observation) for a in areas):
        summaries.append("Moisture impact is not limited to room walls and appears to extend to the parking ceiling area.")

    if any("external wall" in normalize_for_match(a.positive_observation) or "external wall" in normalize_for_match(' '.join(a.related_summary_points)) for a in areas):
        summaries.append("The inspection suggests that part of the problem may be linked to defects in the external wall surface.")

    if any("tile" in normalize_for_match(a.positive_observation) or "plumbing" in normalize_for_match(a.positive_observation) for a in areas):
        summaries.append("Wet-area defects such as open tile joints, hollowness, or plumbing-related issues appear to be contributing factors.")

    if not summaries:
        summaries.append("The property shows signs of moisture-related distress, but the available evidence is limited.")

    return unique_preserve_order(summaries)


def build_root_cause_summary(areas: List[AreaObservation]) -> List[str]:
    causes = []
    for area in areas:
        causes.extend(area.probable_root_cause)
    return unique_preserve_order(causes)


def detect_conflicts(areas: List[AreaObservation]) -> List[str]:
    conflicts = []
    for area in areas:
        text = normalize_for_match(
            f"{area.negative_observation} {area.positive_observation} {' '.join(area.supporting_checklist)}"
        )
        if "concealed plumbing" in text and "external wall crack" in text:
            conflicts.append(
                f"For {area.area_name}, both plumbing-related leakage and external-wall ingress indicators are present; the primary source cannot be isolated with certainty from the documents alone."
            )
    return unique_preserve_order(conflicts)


def collect_global_missing(ctx: PropertyContext, areas: List[AreaObservation]) -> List[str]:
    missing = []

    for field_name, value in asdict(ctx).items():
        title = field_name.replace("_", " ").title()
        if isinstance(value, list):
            if not value:
                missing.append(f"{title}: Not Available")
        else:
            if not value or value == "Not Available":
                missing.append(f"{title}: Not Available")

    if not any(a.related_thermal_pages for a in areas):
        missing.append("No thermal pages could be confidently linked to area-wise observations")

    return unique_preserve_order(missing)



In [15]:
def write_json_report(ddr: DDRResult, output_path: str) -> None:
    payload = {
        "property_context": asdict(ddr.property_context),
        "property_issue_summary": ddr.property_issue_summary,
        "area_wise_observations": [asdict(a) for a in ddr.area_wise_observations],
        "probable_root_cause_summary": ddr.probable_root_cause_summary,
        "additional_notes": ddr.additional_notes,
        "missing_or_unclear_information": ddr.missing_or_unclear_information,
        "conflicts": ddr.conflicts,
    }

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)


In [16]:
def build_styles():
    styles = getSampleStyleSheet()

    styles.add(ParagraphStyle(
        name="CoverTitle",
        parent=styles["Title"],
        fontName="Helvetica-Bold",
        fontSize=24,
        leading=30,
        alignment=TA_CENTER,
        textColor=colors.HexColor("#2F3E46"),
        spaceAfter=14,
    ))

    styles.add(ParagraphStyle(
        name="CoverSub",
        parent=styles["Normal"],
        fontName="Helvetica",
        fontSize=11,
        leading=15,
        alignment=TA_CENTER,
        textColor=colors.HexColor("#5C6770"),
        spaceAfter=8,
    ))

    styles.add(ParagraphStyle(
        name="SectionTitleDDR",
        parent=styles["Heading1"],
        fontName="Helvetica-Bold",
        fontSize=15,
        leading=20,
        textColor=colors.white,
        backColor=colors.HexColor("#C79A2B"),
        borderPadding=(6, 8, 6),
        spaceBefore=10,
        spaceAfter=10,
    ))

    styles.add(ParagraphStyle(
        name="SubTitleDDR",
        parent=styles["Heading2"],
        fontName="Helvetica-Bold",
        fontSize=12,
        leading=16,
        textColor=colors.HexColor("#243238"),
        spaceBefore=8,
        spaceAfter=6,
    ))

    styles.add(ParagraphStyle(
        name="BodyDDR",
        parent=styles["BodyText"],
        fontName="Helvetica",
        fontSize=9.5,
        leading=14,
        textColor=colors.HexColor("#1F2933"),
        spaceAfter=4,
    ))

    styles.add(ParagraphStyle(
        name="SmallDDR",
        parent=styles["BodyText"],
        fontName="Helvetica",
        fontSize=8.5,
        leading=12,
        textColor=colors.HexColor("#4A5568"),
        spaceAfter=3,
    ))

    styles.add(ParagraphStyle(
        name="BulletDDR",
        parent=styles["BodyText"],
        fontName="Helvetica",
        fontSize=9.5,
        leading=14,
        leftIndent=14,
        firstLineIndent=-8,
        bulletIndent=0,
        textColor=colors.HexColor("#1F2933"),
        spaceAfter=2,
    ))

    styles.add(ParagraphStyle(
        name="CaptionDDR",
        parent=styles["Italic"],
        fontName="Helvetica-Oblique",
        fontSize=8,
        leading=10,
        alignment=TA_CENTER,
        textColor=colors.HexColor("#5C6770"),
        spaceAfter=6,
    ))

    return styles


def draw_page_background(canvas, doc):
    width, height = A4

    canvas.saveState()
    canvas.setFillColor(colors.HexColor("#2F343A"))
    canvas.rect(0, height - 28, width, 28, fill=1, stroke=0)

    canvas.setFillColor(colors.HexColor("#C79A2B"))
    canvas.rect(0, height - 32, width, 4, fill=1, stroke=0)

    canvas.setFillColor(colors.HexColor("#C79A2B"))
    canvas.rect(0, 0, width, 10, fill=1, stroke=0)

    canvas.setFillColor(colors.white)
    canvas.setFont("Helvetica-Bold", 9)
    canvas.drawString(2 * cm, height - 19, "Detailed Diagnosis Report")

    canvas.setFillColor(colors.HexColor("#5C6770"))
    canvas.setFont("Helvetica", 8)
    canvas.drawRightString(width - 2 * cm, 18, f"Page {doc.page}")
    canvas.restoreState()


def fit_image(path, max_w, max_h):
    reader = ImageReader(path)
    iw, ih = reader.getSize()
    scale = min(max_w / iw, max_h / ih)
    width = iw * scale
    height = ih * scale
    return width, height


def bullet_list(items, styles):
    flow = []
    if not items:
        flow.append(Paragraph("• Not Available", styles["BulletDDR"]))
        return flow
    for item in items:
        flow.append(Paragraph(f"• {html_escape(item)}", styles["BulletDDR"]))
    return flow


def make_info_table(ctx: PropertyContext, styles):
    rows = [
        ["Customer Name", ctx.customer_name],
        ["Mobile", ctx.mobile],
        ["Email", ctx.email],
        ["Address", ctx.address],
        ["Property Age (Years)", ctx.property_age_years],
        ["Floors", ctx.floors],
        ["Inspection Date", ctx.inspection_date],
        ["Inspected By", ctx.inspected_by],
        ["Property Type", ctx.property_type],
        ["Impacted Areas/Rooms", ", ".join(ctx.impacted_rooms) if ctx.impacted_rooms else "Not Available"],
    ]

    table_data = []
    for k, v in rows:
        table_data.append([
            Paragraph(f"<b>{html_escape(k)}</b>", styles["BodyDDR"]),
            Paragraph(html_escape(v), styles["BodyDDR"])
        ])

    tbl = Table(table_data, colWidths=[4.7 * cm, 11.5 * cm])
    tbl.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#EAEFF2")),
        ("BACKGROUND", (0, 0), (0, -1), colors.HexColor("#F7F9FA")),
        ("BOX", (0, 0), (-1, -1), 0.5, colors.HexColor("#B0BEC5")),
        ("INNERGRID", (0, 0), (-1, -1), 0.35, colors.HexColor("#CFD8DC")),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("LEFTPADDING", (0, 0), (-1, -1), 8),
        ("RIGHTPADDING", (0, 0), (-1, -1), 8),
        ("TOPPADDING", (0, 0), (-1, -1), 6),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 6),
    ]))
    return tbl


def make_area_header(area: AreaObservation, styles):
    severity_color = {
        "High": "#D64545",
        "Medium": "#D18F00",
        "Low": "#2E7D32",
    }.get(area.severity_level, "#607D8B")

    data = [[
        Paragraph(
            f"<b>Impacted Area {html_escape(area.impacted_area_no)}: {html_escape(area.area_name)}</b>",
            styles["SubTitleDDR"]
        ),
        Paragraph(
            f'<font color="{severity_color}"><b>Severity: {html_escape(area.severity_level)}</b></font>',
            styles["BodyDDR"]
        )
    ]]
    tbl = Table(data, colWidths=[12.2 * cm, 4 * cm])
    tbl.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, -1), colors.HexColor("#F2F5F7")),
        ("BOX", (0, 0), (-1, -1), 0.4, colors.HexColor("#CBD5DC")),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("LEFTPADDING", (0, 0), (-1, -1), 8),
        ("RIGHTPADDING", (0, 0), (-1, -1), 8),
        ("TOPPADDING", (0, 0), (-1, -1), 6),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 6),
        ("ALIGN", (1, 0), (1, 0), "RIGHT"),
    ]))
    return tbl



In [17]:
def write_pdf_report(ddr: DDRResult, output_path: str):
    styles = build_styles()

    doc = SimpleDocTemplate(
        output_path,
        pagesize=A4,
        leftMargin=1.7 * cm,
        rightMargin=1.7 * cm,
        topMargin=2.2 * cm,
        bottomMargin=1.6 * cm,
        title="Detailed Diagnosis Report",
    )

    story = []

    # Cover page
    story.append(Spacer(1, 3.2 * cm))
    story.append(Paragraph("Detailed Diagnosis Report", styles["CoverTitle"]))
    story.append(Spacer(1, 0.2 * cm))
    story.append(Paragraph("Generated from Inspection Report and Thermal Report", styles["CoverSub"]))
    story.append(Spacer(1, 0.35 * cm))
    story.append(Paragraph(
        f"Inspection Date: {html_escape(ddr.property_context.inspection_date)}",
        styles["CoverSub"]
    ))
    story.append(Paragraph(
        f"Property Type: {html_escape(ddr.property_context.property_type)}",
        styles["CoverSub"]
    ))
    story.append(Spacer(1, 1.2 * cm))

    cover_tbl = Table([
        [Paragraph("<b>Prepared DDR Output</b>", styles["BodyDDR"])],
        [Paragraph("This report combines inspection observations and thermal evidence in a clear client-friendly format.", styles["BodyDDR"])]
    ], colWidths=[15.8 * cm])
    cover_tbl.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#C79A2B")),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
        ("BACKGROUND", (0, 1), (-1, 1), colors.HexColor("#F8F5EE")),
        ("BOX", (0, 0), (-1, -1), 0.6, colors.HexColor("#D3C3A0")),
        ("LEFTPADDING", (0, 0), (-1, -1), 10),
        ("RIGHTPADDING", (0, 0), (-1, -1), 10),
        ("TOPPADDING", (0, 0), (-1, -1), 8),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 8),
    ]))
    story.append(cover_tbl)
    story.append(PageBreak())

    # Section 1
    story.append(Paragraph("1. Property Issue Summary", styles["SectionTitleDDR"]))
    story.append(make_info_table(ddr.property_context, styles))
    story.append(Spacer(1, 0.35 * cm))
    for p in ddr.property_issue_summary:
        story.append(Paragraph(f"• {html_escape(p)}", styles["BulletDDR"]))

    # Area-wise observations
    story.append(Spacer(1, 0.35 * cm))
    story.append(Paragraph("2. Area-wise Observations", styles["SectionTitleDDR"]))

    for area in ddr.area_wise_observations:
        block = []
        block.append(make_area_header(area, styles))
        block.append(Spacer(1, 0.15 * cm))

        block.append(Paragraph("<b>Observed Issue</b>", styles["SubTitleDDR"]))
        block.append(Paragraph(html_escape(area.negative_observation), styles["BodyDDR"]))

        block.append(Paragraph("<b>Related Source / Exposure Observation</b>", styles["SubTitleDDR"]))
        block.append(Paragraph(html_escape(area.positive_observation), styles["BodyDDR"]))

        block.append(Paragraph("<b>Linked Summary Evidence</b>", styles["SubTitleDDR"]))
        block.extend(bullet_list(area.related_summary_points or ["Not Available"], styles))

        block.append(Paragraph("<b>Supporting Checklist Evidence</b>", styles["SubTitleDDR"]))
        block.extend(bullet_list(area.supporting_checklist or ["Not Available"], styles))

        block.append(Paragraph("<b>Thermal Evidence</b>", styles["SubTitleDDR"]))
        if area.related_thermal_pages:
            for t in area.related_thermal_pages:
                thermal_text = (
                    f"Thermal page {t.page_number}: image {t.thermal_image_name}, "
                    f"date {t.date}, hotspot {t.hotspot_c}, coldspot {t.coldspot_c}"
                )
                block.append(Paragraph(f"• {html_escape(thermal_text)}", styles["BulletDDR"]))
        else:
            block.append(Paragraph("• Not Available", styles["BulletDDR"]))

        block.append(Paragraph("<b>Probable Root Cause</b>", styles["SubTitleDDR"]))
        block.extend(bullet_list(area.probable_root_cause or ["Not Available"], styles))

        block.append(Paragraph("<b>Severity Assessment</b>", styles["SubTitleDDR"]))
        block.append(Paragraph(f"<b>Severity:</b> {html_escape(area.severity_level)}", styles["BodyDDR"]))
        block.append(Paragraph(f"<b>Reasoning:</b> {html_escape(area.severity_reasoning)}", styles["BodyDDR"]))

        block.append(Paragraph("<b>Recommended Actions</b>", styles["SubTitleDDR"]))
        block.extend(bullet_list(area.recommended_actions or ["Not Available"], styles))

        block.append(Paragraph("<b>Additional Notes</b>", styles["SubTitleDDR"]))
        block.extend(bullet_list(area.additional_notes or ["Not Available"], styles))

        block.append(Paragraph("<b>Missing or Unclear Information</b>", styles["SubTitleDDR"]))
        block.extend(bullet_list(area.missing_or_unclear_information or ["Not Available"], styles))

        block.append(Paragraph("<b>Image Evidence</b>", styles["SubTitleDDR"]))
        if area.related_images:
            for img_ref in area.related_images[:3]:
                if os.path.exists(img_ref.image_path):
                    w, h = fit_image(img_ref.image_path, MAX_IMAGE_WIDTH, MAX_IMAGE_HEIGHT)
                    block.append(Image(img_ref.image_path, width=w, height=h))
                    block.append(Paragraph(html_escape(img_ref.caption), styles["CaptionDDR"]))
        else:
            block.append(Paragraph("Image Not Available", styles["BodyDDR"]))

        block.append(Spacer(1, 0.35 * cm))
        story.append(KeepTogether(block))
        story.append(Spacer(1, 0.15 * cm))

    # Global sections
    story.append(PageBreak())
    story.append(Paragraph("3. Probable Root Cause Summary", styles["SectionTitleDDR"]))
    story.extend(bullet_list(ddr.probable_root_cause_summary or ["Not Available"], styles))

    story.append(Paragraph("4. Additional Notes", styles["SectionTitleDDR"]))
    story.extend(bullet_list(ddr.additional_notes or ["Not Available"], styles))

    story.append(Paragraph("5. Missing or Unclear Information", styles["SectionTitleDDR"]))
    story.extend(bullet_list(ddr.missing_or_unclear_information or ["Not Available"], styles))

    story.append(Paragraph("6. Conflicts", styles["SectionTitleDDR"]))
    story.extend(bullet_list(ddr.conflicts or ["Not Available"], styles))

    doc.build(story, onFirstPage=draw_page_background, onLaterPages=draw_page_background)



In [18]:
def generate_ddr_pdf(
    thermal_pdf: str = THERMAL_PDF,
    inspection_pdf: str = INSPECTION_PDF,
    output_dir: str = OUTPUT_DIR,
) -> DDRResult:
    ensure_dir(output_dir)

    thermal_img_dir = os.path.join(output_dir, THERMAL_IMG_DIR)
    inspection_img_dir = os.path.join(output_dir, INSPECTION_IMG_DIR)

    thermal_img_map = export_page_renderings(thermal_pdf, thermal_img_dir, "thermal") if EXPORT_ALL_PAGE_IMAGES else {}
    inspection_img_map = export_page_renderings(inspection_pdf, inspection_img_dir, "inspection") if EXPORT_ALL_PAGE_IMAGES else {}

    thermal_pages = read_pdf_pages(thermal_pdf)
    inspection_pages = read_pdf_pages(inspection_pdf)

    property_context = extract_property_context(inspection_pages)
    areas = extract_impacted_area_blocks(inspection_pages, inspection_img_map)
    summary_map = extract_summary_pairs(inspection_pages)
    checklist_evidence = extract_checklist_evidence(inspection_pages)
    thermal_observations = extract_thermal_observations(thermal_pages, thermal_img_map)

    attach_summary_to_areas(areas, summary_map)
    attach_checklists_to_areas(areas, checklist_evidence)
    attach_thermal_to_areas(areas, thermal_observations)
    fill_reasoning(areas)

    property_issue_summary = build_property_summary(areas)
    probable_root_cause_summary = build_root_cause_summary(areas)
    conflicts = detect_conflicts(areas)
    global_missing = collect_global_missing(property_context, areas)

    ddr = DDRResult(
        property_context=property_context,
        property_issue_summary=property_issue_summary,
        area_wise_observations=areas,
        probable_root_cause_summary=probable_root_cause_summary,
        additional_notes=[
            "This generated DDR is based only on the content found in the supplied PDFs.",
            "Where the documents do not clearly identify the exact room-image mapping, the report keeps the statement conservative.",
            "The report is structured for client-friendly reading and evidence support.",
        ],
        missing_or_unclear_information=global_missing,
        conflicts=conflicts,
    )

    write_json_report(ddr, os.path.join(output_dir, OUTPUT_JSON))
    write_pdf_report(ddr, os.path.join(output_dir, OUTPUT_PDF))

    return ddr


In [19]:
missing = [p for p in [THERMAL_PDF, INSPECTION_PDF] if not os.path.exists(p)]
if missing:
    raise FileNotFoundError(
        "Missing required input files: " + ", ".join(missing) +
        ". Upload 'Thermal Images.pdf' and 'Sample Report.pdf' into Colab first."
    )

result = generate_ddr_pdf()

print(f"DDR PDF generated successfully in: {os.path.abspath(OUTPUT_DIR)}")
print(f"Areas processed: {len(result.area_wise_observations)}")
print(f"PDF report: {os.path.join(OUTPUT_DIR, OUTPUT_PDF)}")
print(f"JSON report: {os.path.join(OUTPUT_DIR, OUTPUT_JSON)}")

DDR PDF generated successfully in: /content/ddr_output
Areas processed: 7
PDF report: ddr_output/generated_ddr.pdf
JSON report: ddr_output/structured_ddr.json
